# Computational Tools for Data Science

In [2]:
import json
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

DATASETS = ["business", "checkin", "review", "tip", "user"]
NESTED = {"business": ["attributes", "hours"]}   # dict columns -> JSON strings

def json_to_parquet(name: str, chunksize: int = 200_000) -> Path:
    src = RAW / f"yelp_academic_dataset_{name}.json"
    dst = PROCESSED / f"{name}.parquet"
    reader = pd.read_json(src, lines=True, chunksize=chunksize,
                          dtype=False, convert_dates=False)
    writer, n_rows = None, 0
    try:
        for chunk in reader:
            for col in NESTED.get(name, []):
                chunk[col] = chunk[col].map(
                    lambda x: json.dumps(x) if isinstance(x, dict) else None)
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(dst, table.schema, compression="zstd")
            writer.write_table(table)
            n_rows += len(chunk)
    finally:
        if writer is not None:
            writer.close()
    print(f"{name}: {n_rows:,} rows -> {dst}")
    return dst

for name in DATASETS:
    if not (PROCESSED / f"{name}.parquet").exists():   # skip if already converted
        json_to_parquet(name)

ModuleNotFoundError: No module named 'pyarrow'